# Comparing Gradio Chat Interfaces

This notebook shows three ways to build a chat interface with Gradio:
1. **No SDK** (Plain Python)
2. **OpenAI SDK** (Standard API calls)
3. **LangGraph** (State Machine approach)

In [ ]:
import gradio as gr
from dotenv import load_dotenv
import os

# Load API keys from .env
load_dotenv(override=True)

## 1. No SDK (Plain Python)
Here we just use a simple Python function. No AI is involved. It simply takes the `message` and `history` and returns a string.

In [ ]:
def chat_no_sdk(message, history):
    # We can ignore history for this simple echo bot
    return f"You said: '{message}'. I am a simple Python function, no AI here!"

# Uncomment the line below to run this interface
# gr.ChatInterface(chat_no_sdk, type="messages").launch()

## 2. OpenAI SDK
This is the exact same, simple approach you used in `1_foundations/3_lab3.ipynb`. We use the `openai` Python package, construct the messages, and pass them directly to the LLM.

In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

def chat_openai(message, history):
    # 1. Combine all messages manually
    messages = [{"role": "system", "content": "You are a helpful AI assistant."}] + history + [{"role": "user", "content": message}]
    
    # 2. Call the LLM
    response = client.chat.completions.create(
        model="openai/gpt-4o-mini",
        messages=messages
    )
    
    # 3. Return the string content
    return response.choices[0].message.content

# Uncomment the line below to run this interface
# gr.ChatInterface(chat_openai, type="messages").launch()

## 3. LangGraph
Here we build a state machine. Instead of manually constructing the array inside the chat function, we feed the messages into the **State**. The `ChatOpenAI` node processes the State, and the graph automatically appends the LLM's response to the history.

In [ ]:
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI
from pydantic import BaseModel

# A. Define the State
class State(BaseModel):
    messages: Annotated[list, add_messages]

# B. Initialize Graph Builder
graph_builder = StateGraph(State)

# C. Create Node
llm = ChatOpenAI(
    model="openai/gpt-4o-mini", 
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

def chatbot_node(state: State) -> State:
    # Pass the state's messages to the LLM
    response = llm.invoke(state.messages)
    # Return the new message to be appended
    return State(messages=[response])

graph_builder.add_node("chatbot", chatbot_node)

# D. Create Edges
graph_builder.add_edge(START, "chatbot")
graph_builder.add_edge("chatbot", END)

# E. Compile
graph = graph_builder.compile()

# F. Gradio Chat Function
def chat_langgraph(message, history):
    # Prepare the input for the graph
    messages = [{"role": "system", "content": "You are a helpful AI assistant."}] + history + [{"role": "user", "content": message}]
    
    # Initialize the state and run the graph
    initial_state = State(messages=messages)
    result = graph.invoke(initial_state)
    
    # Extract the final message content
    return result["messages"][-1].content

# Uncomment the line below to run this interface
# gr.ChatInterface(chat_langgraph, type="messages").launch()